# FInal submission 

- applying the exact preprocessing and feature eng pipeine from 02 and 04 notebook to test.csv we'll predict with the single tuned XGBoost chosen in 06 notebook 

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

# --- Constants ---
DATA_DIR = Path("../data")
TEST_CSV = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"
MODELS_DIR = Path("../models")
SUBMISSIONS_DIR = Path("../submissions")

ID_COLUMN = "id"
TARGET_COLUMN = "class"
NUMERIC_FEATURES = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
CATEGORICAL_FEATURES = ["spectral_type", "galaxy_population"]

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

1. load test data 

In [2]:
test_df = pd.read_csv(TEST_CSV)

required_columns = {ID_COLUMN, *NUMERIC_FEATURES, *CATEGORICAL_FEATURES}
missing = required_columns - set(test_df.columns)
assert not missing, f"test.csv is missing expected columns: {missing}"
assert test_df[ID_COLUMN].is_unique, "Duplicate ids found in test.csv"
assert test_df.isnull().sum().sum() == 0, "Unexpected missing values in test.csv"
assert TARGET_COLUMN not in test_df.columns, "test.csv unexpectedly contains the target column"

print(f"Loaded {test_df.shape[0]:,} rows, {test_df.shape[1]} columns — all checks passed")

Loaded 247,435 rows, 11 columns — all checks passed


2. applying preprocessing 

In [3]:
onehot_encoder = joblib.load(MODELS_DIR / "onehot_encoder.joblib")
label_encoder = joblib.load(MODELS_DIR / "label_encoder.joblib")

encoded_array = onehot_encoder.transform(test_df[CATEGORICAL_FEATURES])
encoded_columns = onehot_encoder.get_feature_names_out(CATEGORICAL_FEATURES)
encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=test_df.index)

processed_test_df = pd.concat(
    [test_df[[ID_COLUMN, *NUMERIC_FEATURES]], encoded_df],
    axis=1,
)
print(f"Processed test shape: {processed_test_df.shape}")
processed_test_df.head(2)

Processed test shape: (247435, 15)


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type_A/F,spectral_type_G/K,spectral_type_M,spectral_type_O/B,galaxy_population_Blue_Cloud,galaxy_population_Red_Sequence
0,577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,0.0,1.0,0.0,0.0,0.0,1.0
1,577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,0.0,0.0,1.0,0.0,0.0,1.0


3. apply feature eng

In [4]:
def add_color_index_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add photometric color-index features as magnitude differences.

    Identical formulas to `04_feature_engineering.ipynb`. Returns a new
    DataFrame; input is not mutated.

    Args:
        df: DataFrame containing raw `u`, `g`, `r`, `i`, `z` magnitude columns.

    Returns:
        A copy of `df` with six color-index columns appended.
    """
    result = df.copy()
    result["color_u_g"] = df["u"] - df["g"]
    result["color_g_r"] = df["g"] - df["r"]
    result["color_r_i"] = df["r"] - df["i"]
    result["color_i_z"] = df["i"] - df["z"]
    result["color_u_r"] = df["u"] - df["r"]
    result["color_g_i"] = df["g"] - df["i"]
    return result


def add_redshift_log_feature(df: pd.DataFrame) -> pd.DataFrame:
    """Add a log1p-transformed redshift column alongside the raw one.

    Args:
        df: DataFrame containing a non-negative `redshift` column.

    Returns:
        A copy of `df` with a `redshift_log1p` column appended.
    """
    result = df.copy()
    result["redshift_log1p"] = np.log1p(df["redshift"])
    return result

In [5]:
test_features_df = add_color_index_features(processed_test_df)
test_features_df = add_redshift_log_feature(test_features_df)

assert test_features_df.shape[0] == test_df.shape[0], "Row count changed during feature engineering"
assert test_features_df.isnull().sum().sum() == 0, "Nulls introduced during feature engineering"

print(f"Final test feature shape: {test_features_df.shape}")

Final test feature shape: (247435, 22)


4. validate feature alignment with training data 

In [6]:
train_features_columns = pd.read_csv(
    DATA_DIR / "processed" / "train_features.csv", nrows=0
).columns.tolist()
expected_feature_columns = [
    c for c in train_features_columns if c not in ("id", "class", "class_encoded")
]

test_feature_columns = [c for c in test_features_df.columns if c != ID_COLUMN]

assert test_feature_columns == expected_feature_columns, (
    f"Column mismatch between test features and training features!\n"
    f"Expected: {expected_feature_columns}\n"
    f"Got: {test_feature_columns}"
)
print("Column alignment verified — test features exactly match training features.")

Column alignment verified — test features exactly match training features.


5. predict 

In [7]:
final_model = joblib.load(MODELS_DIR / "xgboost_tuned.joblib")

X_test = test_features_df[expected_feature_columns].to_numpy()
predicted_class_encoded = final_model.predict(X_test)
predicted_class_labels = label_encoder.inverse_transform(predicted_class_encoded)

print(f"Predicted {len(predicted_class_labels):,} rows")
pd.Series(predicted_class_labels).value_counts()

Predicted 247,435 rows


GALAXY    162047
QSO        50105
STAR       35283
Name: count, dtype: int64

6. build + validate submission file

In [8]:
submission_df = pd.DataFrame({ID_COLUMN: test_df[ID_COLUMN], TARGET_COLUMN: predicted_class_labels})

sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV)

assert list(submission_df.columns) == list(sample_submission_df.columns), "Column name/order mismatch"
assert len(submission_df) == len(sample_submission_df), "Row count mismatch"
assert set(submission_df[ID_COLUMN]) == set(sample_submission_df[ID_COLUMN]), "id set mismatch"
assert submission_df[TARGET_COLUMN].isin(label_encoder.classes_).all(), "Unexpected class label produced"

print("All submission format checks passed.")
submission_df.head()

All submission format checks passed.


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


In [9]:
submission_output_path = SUBMISSIONS_DIR / "submission.csv"
submission_df.to_csv(submission_output_path, index=False)
print(f"Saved final submission to {submission_output_path}")

Saved final submission to ..\submissions\submission.csv


## Project Summary

| Stage | Notebook | Key Result |
|---|---|---|
| EDA | `01_eda.ipynb` | Moderate class imbalance (65/20/14%); `redshift` most discriminative feature; no data quality issues |
| Preprocessing | `02_preprocessing.ipynb` | Target + categorical encoding; numeric scaling deliberately deferred to per-fold pipelines |
| Baselines | `03_baseline_models.ipynb` | XGBoost best of 4 models (0.9564 CV macro F1); `STAR`→`GALAXY` confusion identified as the dominant error mode |
| Feature Engineering | `04_feature_engineering.ipynb` | Color-index features became XGBoost's top predictor by gain, but CV score barely moved (0.9556) — features substituted for signal the model already captured |
| Hyperparameter Tuning | `05_optuna_tuning.ipynb` | 40-trial Optuna search recovered the lift back to 0.9563, at roughly 2x training cost |
| Ensembling | `06_ensemble.ipynb` | No combination strategy beat the single tuned model — shipped the single model instead of unnecessary complexity |
| **Final Submission** | `07_final_submission.ipynb` | Single tuned XGBoost, full pipeline validated end-to-end against training-time schema |

**The `STAR`↔`GALAXY` confusion at low redshift was never fully resolved** across feature engineering, tuning, or ensembling — consistent evidence (not a single model's blind spot, but a property of the data) that this specific boundary is genuinely ambiguous from photometry alone. A clearly stated limitation, documented rather than hidden, and a natural candidate for future work if additional data (e.g. imaging morphology) becomes available.